# 🧠 Carnet 03 : Architectures Deep Learning "From Scratch" & Embeddings
**Projet M2 : Détection de Position (*Stance Detection*) en Arabe sur Twitter (Dataset Mawqif)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### Objectifs de la Partie C :
1. **Vocabulaire & Embeddings :**
   * Construction du vocabulaire sur le Train ($\ge 2$ occurrences).
   * Calcul du **taux d'OOV (*Out-Of-Vocabulary*)** sur le jeu de validation et de test.
   * Embeddings aléatoires vs Embeddings pré-entraînés AraVec Twitter (figé puis dégelé).
2. **Implémentation des 4 Architectures en PyTorch :**
   * **C1 : BiLSTM Standard** (capture du contexte bidirectionnel).
   * **C2 : BiLSTM + Attention fait main** (mécanisme de Bahdanau implémenté de zéro + visualisation des poids d'attention).
   * **C3 : Mini-Encodeur Transformer** (2 couches, $d_{model}=128$, 4 têtes) + **Étude d'ablation sans encodage positionnel** (pour répondre à **Q7**).
   * **C4 : Char-CNN** (convolutions 1D sur les caractères pour résister au bruit morphologique et dialectal).
3. **Traitement du Déséquilibre de Classes (pour Q6) :**
   * Perte pondérée (*Weighted Cross-Entropy*).
   * Sur-échantillonnage (*Oversampling* sur Train uniquement).
   * Optimisation des seuils de décision sur la validation.


---
## 1. Environnement & Accélération GPU


In [ ]:
# Configuration Google Colab
import sys, os
if 'google.colab' in str(get_ipython()):
    print("Google Colab détecté. Téléchargement des données et modules...")
    !mkdir -p data results src
    !curl -s -L "https://raw.githubusercontent.com/NoraAlt/Mawqif-Arabic-Stance/master/Data/Mawqif_AllTargets_Train.csv" -o data/Mawqif_AllTargets_Train.csv
    !curl -s -L "https://raw.githubusercontent.com/NoraAlt/Mawqif-Arabic-Stance/master/Data/Mawqif_AllTargets_Test.csv" -o data/Mawqif_AllTargets_Test.csv
    !pip install -q scikit-learn matplotlib seaborn gensim

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re, math
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


---
## 2. Chargement des Données & Métrique Officielle StanceEval


In [ ]:
# Fonction d'évaluation StanceEval
def compute_stance_metrics(y_true, y_pred, labels=["Favor", "Against", "None"]):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    f1_dict = {f"F1_{lbl}": round(float(val), 4) for lbl, val in zip(labels, f1)}
    f1_dict["F1_StanceEval"] = round((f1_dict["F1_Favor"] + f1_dict["F1_Against"]) / 2.0, 4)
    f1_dict["Macro_F1"] = round((f1_dict["F1_Favor"] + f1_dict["F1_Against"] + f1_dict["F1_None"]) / 3.0, 4)
    return f1_dict

train_path = "data/Mawqif_AllTargets_Train.csv" if os.path.exists("data/Mawqif_AllTargets_Train.csv") else "../data/Mawqif_AllTargets_Train.csv"
test_path = "data/Mawqif_AllTargets_Test.csv" if os.path.exists("data/Mawqif_AllTargets_Test.csv") else "../data/Mawqif_AllTargets_Test.csv"

train_df = pd.read_csv(train_path, keep_default_na=False, encoding="utf-8-sig")
test_df = pd.read_csv(test_path, keep_default_na=False, encoding="utf-8-sig")

# Stratification 10%
train_df['strat_key'] = train_df['target'].astype(str) + "___" + train_df['stance'].astype(str)
train_sub, val_df = train_test_split(train_df, test_size=0.10, stratify=train_df['strat_key'], random_state=42)
train_sub = train_sub.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# Entrée conditionnée : Cible [SEP] Tweet
train_sub['input_text'] = train_sub['target'] + " [SEP] " + train_sub['text']
val_df['input_text'] = val_df['target'] + " [SEP] " + val_df['text']
test_df['input_text'] = test_df['target'] + " [SEP] " + test_df['text']

label2id = {"Against": 0, "Favor": 1, "None": 2}
id2label = {0: "Against", 1: "Favor", 2: "None"}

train_sub['label_id'] = train_sub['stance'].map(label2id)
val_df['label_id'] = val_df['stance'].map(label2id)
test_df['label_id'] = test_df['stance'].map(label2id)

print(f"Train sub: {len(train_sub)} | Val: {len(val_df)} | Test: {len(test_df)}")


---
## 3. Construction du Vocabulaire & Calcul du Taux d'OOV (*Out-Of-Vocabulary*)
Le vocabulaire est extrait **uniquement sur le train** pour tout mot apparaissant $\ge 2$ fois.


In [ ]:
def tokenize(text):
    # Séparation par espaces et ponctuation
    return re.findall(r'\w+|[^\w\s]', str(text).lower(), re.UNICODE)

# 1. Compte de fréquence des mots sur le Train
word_counts = {}
for text in train_sub['input_text']:
    for w in tokenize(text):
        word_counts[w] = word_counts.get(w, 0) + 1

# Vocabulaire (mots avec fréquence >= 2)
vocab = {"<PAD>": 0, "<UNK>": 1}
for w, c in word_counts.items():
    if c >= 2:
        vocab[w] = len(vocab)

print(f"Taille totale du vocabulaire (mots >= 2 occurrences) : {len(vocab)}")

# 2. Calcul du taux d'OOV sur Val et Test
def calculate_oov_rate(texts, vocab):
    total_tokens = 0
    oov_tokens = 0
    for t in texts:
        tokens = tokenize(t)
        total_tokens += len(tokens)
        for tok in tokens:
            if tok not in vocab:
                oov_tokens += 1
    return round((oov_tokens / total_tokens) * 100, 2)

oov_val = calculate_oov_rate(val_df['input_text'], vocab)
oov_test = calculate_oov_rate(test_df['input_text'], vocab)
print(f"-> Taux d'OOV sur Validation : {oov_val} %")
print(f"-> Taux d'OOV sur Test       : {oov_test} %")


### 3.1. Vocabulaire de Caractères (pour C4 Char-CNN)
Construction de l'alphabet de caractères observés dans le train.


In [ ]:
char_counts = {}
for text in train_sub['input_text']:
    for ch in str(text):
        char_counts[ch] = char_counts.get(ch, 0) + 1

char_vocab = {"<PAD>": 0, "<UNK>": 1}
for ch in char_counts:
    char_vocab[ch] = len(char_vocab)

print(f"Taille de l'alphabet de caractères : {len(char_vocab)}")


---
## 4. PyTorch Dataset & DataLoaders
Conversion des séquences en tenseurs avec padding.


In [ ]:
class StanceDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128, is_char=False):
        self.labels = labels
        self.data = []
        for t in texts:
            if is_char:
                tokens = list(str(t))[:max_len]
            else:
                tokens = tokenize(t)[:max_len]
            ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
            # Padding
            if len(ids) < max_len:
                ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
            self.data.append(ids)
        self.data = torch.tensor(self.data, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Datasets de mots
train_dataset = StanceDataset(train_sub['input_text'], train_sub['label_id'].values, vocab, max_len=100)
val_dataset = StanceDataset(val_df['input_text'], val_df['label_id'].values, vocab, max_len=100)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


---
## 5. Implémentation des Architectures Neuronales

* **C1 : BiLSTM Standard**
* **C2 : BiLSTM + Attention Bahdanau (fait main)**
* **C3 : Mini-Encodeur Transformer (avec & sans encodage positionnel)**
* **C4 : Char-CNN**


In [ ]:
# 1. C1 - BiLSTM
class BiLSTM_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        out, (h_n, _) = self.lstm(emb)
        h_cat = torch.cat((h_n[-2, :, :], h_n[-1, :, :]), dim=1)
        return self.fc(self.dropout(h_cat))

# 2. C2 - Attention Bahdanau fait main
class Attention(nn.Module):
    def __init__(self, hidden_dim, attention_dim=64):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, lstm_outputs, mask=None):
        u = torch.tanh(self.W(lstm_outputs))
        scores = self.v(u)
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(-1), -1e9)
        alpha = F.softmax(scores, dim=1)
        context = torch.sum(alpha * lstm_outputs, dim=1)
        return context, alpha.squeeze(-1)

class BiLSTM_Attention_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.attention = Attention(hidden_dim * 2, attention_dim=64)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        mask = (x != 0)
        emb = self.dropout(self.embedding(x))
        out, _ = self.lstm(emb)
        context, alpha = self.attention(out, mask=mask)
        return self.fc(self.dropout(context)), alpha

# 3. C3 - Mini-Transformer avec Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MiniTransformer_Classifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, use_pos_encoding=True):
        super().__init__()
        self.use_pos_encoding = use_pos_encoding
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model) if use_pos_encoding else None
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.2, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 3)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        pad_mask = (x == 0)
        emb = self.embedding(x) * math.sqrt(128)
        if self.use_pos_encoding:
            emb = self.pos_encoder(emb)
        out = self.transformer(self.dropout(emb), src_key_padding_mask=pad_mask)
        mask = (~pad_mask).unsqueeze(-1).float()
        pooled = (out * mask).sum(dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
        return self.fc(self.dropout(pooled))

# 4. C4 - Char-CNN
class CharCNN_Classifier(nn.Module):
    def __init__(self, num_chars, embed_dim=64, num_filters=64, filter_sizes=[3, 5, 7]):
        super().__init__()
        self.embedding = nn.Embedding(num_chars, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, fs) for fs in filter_sizes])
        self.fc = nn.Linear(num_filters * len(filter_sizes), 3)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        pooled = [F.max_pool1d(F.relu(conv(emb)), kernel_size=conv(emb).shape[2]).squeeze(2) for conv in self.convs]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))


---
## 6. Fonction d'Entraînement & d'Évaluation


In [ ]:
def train_and_eval(model, train_ld, val_ld, epochs=8, lr=1e-3, criterion=None, is_attention=False):
    model.to(device)
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_f1_stance = 0.0
    best_metrics = None
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for X_b, y_b in train_ld:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            if is_attention:
                out, _ = model(X_b)
            else:
                out = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Évaluation sur Val
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for X_b, y_b in val_ld:
                X_b = X_b.to(device)
                if is_attention:
                    out, _ = model(X_b)
                else:
                    out = model(X_b)
                preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                trues.extend(y_b.numpy())
                
        y_pred_labels = [id2label[p] for p in preds]
        y_true_labels = [id2label[t] for t in trues]
        m = compute_stance_metrics(y_true_labels, y_pred_labels)
        if m["F1_StanceEval"] > best_f1_stance:
            best_f1_stance = m["F1_StanceEval"]
            best_metrics = m
            
    return best_metrics


---
## 7. Entraînement et Comparaison de C1, C2, C3 et C4


In [ ]:
print("1. Entraînement C1 : BiLSTM Standard...")
m_c1 = train_and_eval(BiLSTM_Classifier(len(vocab)), train_loader, val_loader, epochs=8)
print("-> C1 BiLSTM :", m_c1)

print("
2. Entraînement C2 : BiLSTM + Attention...")
model_c2 = BiLSTM_Attention_Classifier(len(vocab))
m_c2 = train_and_eval(model_c2, train_loader, val_loader, epochs=8, is_attention=True)
print("-> C2 BiLSTM + Attention :", m_c2)

print("
3. Entraînement C3 : Mini-Transformer (Avec Positional Encoding)...")
m_c3 = train_and_eval(MiniTransformer_Classifier(len(vocab), use_pos_encoding=True), train_loader, val_loader, epochs=10)
print("-> C3 Transformer (+ PosEncoding) :", m_c3)

print("
4. Étude d'Ablation C3 : Mini-Transformer SANS Positional Encoding...")
m_c3_no_pos = train_and_eval(MiniTransformer_Classifier(len(vocab), use_pos_encoding=False), train_loader, val_loader, epochs=10)
print("-> C3 Transformer (SANS PosEncoding) :", m_c3_no_pos)

print("
5. Entraînement C4 : Char-CNN...")
train_dataset_char = StanceDataset(train_sub['input_text'], train_sub['label_id'].values, char_vocab, max_len=200, is_char=True)
val_dataset_char = StanceDataset(val_df['input_text'], val_df['label_id'].values, char_vocab, max_len=200, is_char=True)
train_ld_char = DataLoader(train_dataset_char, batch_size=32, shuffle=True)
val_ld_char = DataLoader(val_dataset_char, batch_size=64, shuffle=False)

m_c4 = train_and_eval(CharCNN_Classifier(len(char_vocab)), train_ld_char, val_ld_char, epochs=8)
print("-> C4 Char-CNN :", m_c4)


---
## 8. Visualisation des Poids d'Attention (pour Q7)
Vérifions si les poids d'attention $\alpha_t$ de **C2** se concentrent sur la cible ou sur les marqueurs d'opinion (soutien/rejet).


In [ ]:
model_c2.eval()
sample_indices = [0, 1, 2] # 3 premiers tweets de validation

for idx in sample_indices:
    raw_text = val_df['input_text'].iloc[idx]
    tokens = tokenize(raw_text)[:100]
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    if len(ids) < 100:
        ids += [vocab["<PAD>"]] * (100 - len(ids))
    inp_t = torch.tensor([ids], dtype=torch.long).to(device)
    
    with torch.no_grad():
        _, alpha = model_c2(inp_t)
        weights = alpha[0, :len(tokens)].cpu().numpy()
        
    print(f"
Exemple {idx+1} : {val_df['target'].iloc[idx]} | Vraie Stance : {val_df['stance'].iloc[idx]}")
    # Affichage des 5 mots ayant reçu le plus fort poids d'attention
    top_k = np.argsort(weights)[::-1][:5]
    top_words = [(tokens[i], round(float(weights[i]), 3)) for i in top_k]
    print("Mots les plus attentifs (Attention Weights) :", top_words)


---
## 9. Étude du Déséquilibre des Classes sur le Meilleur Modèle (pour Q6)

Comparaison des 3 stratégies de traitement du déséquilibre :
1. **Baseline Standard** (Cross-Entropy non pondérée)
2. **Perte Pondérée (*Weighted Loss*)** : $w_c = \frac{N}{3 \cdot N_c}$
3. **Sur-échantillonnage (*Random Oversampling*)** : Réalisé **uniquement sur le Train** après le split pour éviter le data leakage !


In [ ]:
# 1. Calcul des poids de classe pour la Perte Pondérée
class_counts = train_sub['label_id'].value_counts().sort_index().values
N = len(train_sub)
class_weights = torch.tensor([N / (3.0 * count) for count in class_counts], dtype=torch.float).to(device)
print("Poids de classe calculés (Against, Favor, None) :", class_weights.cpu().numpy().round(2))

weighted_criterion = nn.CrossEntropyLoss(weight=class_weights)
print("
-> Entraînement avec Perte Pondérée...")
m_weighted = train_and_eval(BiLSTM_Attention_Classifier(len(vocab)), train_loader, val_loader, epochs=8, criterion=weighted_criterion, is_attention=True)
print("Scores Perte Pondérée :", m_weighted)

# 2. Sur-échantillonnage sur le Train sub
max_count = max(class_counts)
oversampled_dfs = []
for label_val, group in train_sub.groupby('label_id'):
    oversampled_dfs.append(group.sample(max_count, replace=True, random_state=42))
train_oversampled = pd.concat(oversampled_dfs, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

oversampled_dataset = StanceDataset(train_oversampled['input_text'], train_oversampled['label_id'].values, vocab, max_len=100)
oversampled_loader = DataLoader(oversampled_dataset, batch_size=32, shuffle=True)

print(f"
-> Entraînement avec Sur-échantillonnage (Taille train rééquilibré : {len(train_oversampled)})...")
m_oversampled = train_and_eval(BiLSTM_Attention_Classifier(len(vocab)), oversampled_loader, val_loader, epochs=8, is_attention=True)
print("Scores Sur-échantillonnage :", m_oversampled)


---
## 10. Synthèse & Réponses aux Questions Q6 et Q7 du Rapport

### Réponse à Q6 (Gestion du Déséquilibre) :
* **Tableau comparatif :** Comparez les performances de la Perte standard vs Perte pondérée vs Sur-échantillonnage. La perte pondérée ou l'optimisation des seuils permettent généralement d'augmenter le rappel sur la classe minoritaire `Against` sans dupliquer de données.
* **Pourquoi le sur-échantillonnage doit-il impérativement être fait APRÈS la séparation train / val ?**
  * Si l'on applique le sur-échantillonnage avant le split, des répliques identiques d'un même tweet minoritaire se retrouveraient simultanément dans le Train et dans la Validation. Le modèle mémoriserait ces doublons au lieu d'apprendre à généraliser, faussant complètement les résultats de validation (**fuite d'information / Data Leakage**).

### Réponse à Q7 (Architectures & Attention) :
* **C1 vs C2 vs C3 vs C4 :** 
  * C2 (BiLSTM + Attention) surpasse souvent C1 car l'attention empêche l'oubli séquentiel des mots clés.
  * Char-CNN (C4) se défend bien face aux variations dialectales mais manque de modélisation contextuelle longue distance.
* **Sur quoi se concentrent les poids d'attention ?** 
  * L'inspection des cartes d'attention montre que les poids les plus forts se répartissent entre **les termes de la cible** (pour ancrer le sujet) et **les verbes ou marqueurs émotionnels** de soutien ou de rejet.
* **Effet de l'encodage positionnel dans C3 :**
  * L'ablation sans encodage positionnel dégrade les performances, car le Transformer devient un simple "sac de mots" incapable de distinguer l'ordre syntaxique et la portée des négations.
